In [ ]:
%cd ../..

In [ ]:
from pathlib import Path

import polars as pl

# Read raw data

In [ ]:
paths_1 = Path("data/raw/biowaste/Apr_1").glob("*.csv")
paths_2 = Path("data/raw/biowaste").glob("*.xlsx")

paths = list(paths_1) + list(paths_2)

cols = ['pcs', 'date', 'meal', 'waste', 'restaurant']

df_list = []

for path in paths:
    df = (
        pl.read_csv(path, separator=";")
        if path.suffix == ".csv"
        else pl.read_excel(path)
    )

    df.columns = cols
    df = df.with_columns(
        pl.lit(path.name).alias('src'),
        pl.col('waste').cast(pl.Float32),
    )

    df_list.append(df)

waste_raw = pl.concat(df_list)
waste_raw.head()

In [ ]:
path_3 = Path("data/raw/biowaste/Biowaste.csv")

cols = ['date', 'restaurant', 'waste_customer_kg', 'waste_coffee_kg', 'waste_kitchen_kg', 'waste_hall_kg', 'src']

waste_raw_2 = (
    pl.read_csv(path_3, separator=';')
    .with_columns(
        pl.lit(path_3.name).alias('src')
    )
)
waste_raw_2.columns = cols

waste_raw_2.head()

In [ ]:
path = "data/processed/dim_restaurants.xlsx"
dim_restaurants = pl.read_excel(path)

dim_restaurants.head()

# Process

In [ ]:
cols = ['date', 'restaurant', 'waste', 'src']

## Process each file

In [ ]:
waste_1 = (
    waste_raw
    .with_columns(
        pl.coalesce(
            pl.col('date').str.to_date("%d.%m.%Y", strict=False),
            pl.col('date').str.to_date("%Y-%m-%d", strict=False, exact=False)
        ).alias('date'),
    )

    .join(dim_restaurants, on='restaurant', how='left')
    .drop('restaurant')
    .rename({'restaurant_id': 'restaurant'})

    .select(cols)
)

waste_1.head()

In [ ]:
waste_2 = (
    waste_raw_2

    .with_columns(
        pl.col('date').str.to_date("%d.%m.%Y", strict=False).alias('date'),
        (pl.col('waste_customer_kg')
         + pl.col('waste_coffee_kg')
         + pl.col('waste_kitchen_kg')
         + pl.col('waste_hall_kg')
        ).cast(pl.Float32).alias('waste')
    )
    .drop('waste_customer_kg', 'waste_coffee_kg', 'waste_kitchen_kg', 'waste_hall_kg')

    .join(dim_restaurants, on='restaurant', how='left')
    .drop('restaurant', 'restaurant_short')
    .rename({'restaurant_id': 'restaurant'})

    .select(cols)
)

waste_2.head()

## Combine into single fact table

In [ ]:
cols = ['id', 'date', 'restaurant', 'waste', 'src']

waste = (
    pl.concat([waste_1, waste_2])


    # For each date-restaurant group, keep entry having greatest 'waste'
    .with_columns(
        pl.col('waste').rank(method='ordinal', descending=True).over(['date', 'restaurant']).alias('rank')
    )
    .filter(pl.col('rank') == 1)
    .drop('rank')


    # Add id
    .with_columns(
        pl.concat_str(
            pl.col('date').dt.to_string(),
            pl.col('restaurant').cast(pl.String),
            separator='|'
            
        ).alias('id')
    )

    .select(cols)
)

## Sanity check

In [ ]:
assert waste.group_by('id').len().filter(pl.col('len') != 1).shape[0] == 0

# Save

In [ ]:
path = "data/processed/waste.parquet"

waste.write_parquet(path)